# HW2: Regression and Classification -- Error Analysis
### Video Game Sales 2024 (VGChartz)

**Course:** Introduction to Data Science &nbsp;&nbsp;|&nbsp;&nbsp;

---

## Overview

This notebook continues the exploratory analysis performed in HW1 on the *Video Game Sales 2024*
dataset and moves from description to **prediction and systematic error analysis**. Two supervised
problems are defined on the same feature set:

| Problem | Target | Type |
|---|---|---|
| Regression | `log1p(total_sales)` -- log-transformed global units sold (millions) | continuous |
| Classification | `is_hit` -- whether a title sold at least 1.0M units | binary, imbalanced (~8% positive) |

Every model is evaluated with **k-fold cross-validation**, and all analysis below is performed on
**out-of-fold predictions**, so that no observation is ever analysed by a model that saw it during
training. The notebook is organised to follow the assignment brief:

* **Part A** -- data preparation, feature set, and the choice of `k`
* **Part B** -- regression models (brief section 2) and regression error analysis (brief section 1)
* **Part C** -- classification error analysis (brief section 3)
* **Part D** -- final reflection (brief section 4)

## Part A -- Setup, Data Preparation and Cross-Validation Design

### A.1 Environment and global constants

In [1]:
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              HistGradientBoostingRegressor, HistGradientBoostingClassifier)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             confusion_matrix, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score,
                             precision_score, recall_score, f1_score, fbeta_score,
                             matthews_corrcoef, balanced_accuracy_score, brier_score_loss)
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')

RANDOM_STATE = 42                 # single seed for every stochastic component
N_SPLITS = 5                      # number of cross-validation folds (justified in A.5)
HIT_THRESHOLD_MILLIONS = 1.0      # a title is a "hit" if it sold >= 1.0M units
REFERENCE_YEAR = 2024             # dataset snapshot year, used to derive game age
DATA_PATH = 'vgchartz-2024.csv'

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 11, 'axes.labelsize': 10})
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

ModuleNotFoundError: No module named 'sklearn'

### A.2 Problem formulation and leakage control

**Target choice.** HW1 established that `total_sales` follows an extreme right-tailed, power-law-like
distribution (skewness 8.77, median 0.12M against a maximum of 20.32M). Regressing on the raw scale
would let a handful of blockbusters dominate the squared-error objective. We therefore model
`log1p(total_sales)`, which compresses the tail while remaining monotonic and invertible
(`expm1`), so predictions can always be mapped back to millions of units.

**Leakage control -- the critical modelling decision.** The regional columns
`na_sales`, `jp_sales`, `pal_sales` and `other_sales` are *components* of the target: HW1 verified
that `total_sales` equals their sum. Including them as features would produce a near-perfect
$R^2$ that measures nothing but an accounting identity. **All four regional columns are therefore
excluded from the feature set**, for both the regression and the classification task.

**Row filter.** Only rows with an observed `total_sales` *and* a parseable `release_date` enter the
modelling frame. `total_sales` is missing for 70.4% of the catalogue -- HW1 showed this missingness
is structural (unreviewed, low-volume titles), not random. This means the modelling population is
already biased towards commercially tracked titles, and every conclusion below is conditional on
that population. This is stated again in the final reflection.

**Placeholder handling.** HW1 identified `'Unknown'` in `publisher` (13.8%) and `developer` (6.9%)
as disguised nulls; they are converted to real `NaN` so they are not encoded as a legitimate,
highly frequent category.

In [ ]:
def load_modeling_frame(csv_path: str = DATA_PATH) -> pd.DataFrame:
    """Load vgchartz-2024, engineer features, and keep only rows with an observed target.

    Returns a frame that carries both supervised targets (`log_total_sales`, `is_hit`)
    alongside the raw columns needed for downstream error attribution.
    """
    raw = pd.read_csv(csv_path).drop_duplicates().reset_index(drop=True)

    # --- temporal features -------------------------------------------------
    raw['release_date'] = pd.to_datetime(raw['release_date'], format='%d/%m/%Y', errors='coerce')
    raw['release_year'] = raw['release_date'].dt.year
    raw['release_month'] = raw['release_date'].dt.month

    # --- cross-row feature: how many platforms did this title ship on? ------
    # Computed on the FULL catalogue (including rows without sales) because it
    # describes the release strategy, not the target.
    raw['platform_count'] = raw['title'].map(raw.groupby('title')['console'].nunique())

    # --- disguised nulls (identified in HW1) --------------------------------
    raw[['publisher', 'developer']] = raw[['publisher', 'developer']].replace('Unknown', np.nan)

    # --- supervised frame ---------------------------------------------------
    modeling = raw[raw['total_sales'].notna() & raw['release_year'].notna()].copy()
    modeling['release_year'] = modeling['release_year'].astype(int)
    modeling['game_age_years'] = REFERENCE_YEAR - modeling['release_year']
    modeling['has_critic_score'] = modeling['critic_score'].notna().astype(int)
    modeling['log_total_sales'] = np.log1p(modeling['total_sales'])
    modeling['is_hit'] = (modeling['total_sales'] >= HIT_THRESHOLD_MILLIONS).astype(int)
    return modeling.reset_index(drop=True)


games = load_modeling_frame()

print(f"Modeling frame: {games.shape[0]:,} rows x {games.shape[1]} columns")
print(f"Regression target  : log1p(total_sales)")
print(f"Classification target: is_hit (total_sales >= {HIT_THRESHOLD_MILLIONS}M) -- "
      f"{games['is_hit'].sum():,} positives ({games['is_hit'].mean():.2%})")
print(f"Release years covered: {games['release_year'].min()}-{games['release_year'].max()}")
print(f"critic_score missing within the modeling frame: {games['critic_score'].isna().mean():.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

sns.histplot(games['total_sales'], bins=80, ax=axes[0], color='#4C72B0')
axes[0].set_yscale('log')
axes[0].set_title(f"Raw target: total_sales (skew = {stats.skew(games['total_sales']):.2f})")
axes[0].set_xlabel('Global sales (millions of units)')
axes[0].set_ylabel('Count (log scale)')

sns.histplot(games['log_total_sales'], bins=80, ax=axes[1], color='#DD8452')
axes[1].set_title(f"Modelled target: log1p(total_sales) (skew = {stats.skew(games['log_total_sales']):.2f})")
axes[1].set_xlabel('log1p(global sales)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

**Reading the transform.** The raw target needs a logarithmic *count* axis just to be visible --
the mass is crushed against zero with a tail reaching 20M units. `log1p` reduces skewness from
roughly 8.8 to well under 2 and produces a distribution a squared-error learner can work with.
The transform does not make the problem easy: as Part B shows, the residuals of the log-target model
are *still* right-skewed and heavy-tailed, because the underlying process is genuinely power-law.

### A.3 Feature set

Thirteen source columns collapse into three families, all of which are available *before* a game's
sales are known -- so the feature set is legitimate for a forecasting use-case:

| Family | Columns | Encoding |
|---|---|---|
| Numeric | `critic_score`, `release_year`, `release_month`, `game_age_years`, `platform_count`, `has_critic_score` | median imputation (+ standardisation for scale-sensitive models) |
| Low-cardinality categorical | `genre` (20 levels), `console` (39 levels) | one-hot, rare levels (< 30 occurrences) folded into an `infrequent` bucket |
| High-cardinality categorical | `publisher` (739 levels), `developer` (2,867 levels), `console` | frequency encoding |

Two design notes:

* **`has_critic_score` is a feature, not a nuisance.** 78.1% of the modelling frame has no critic
  score. HW1 showed this missingness is structured -- unreviewed titles are systematically smaller.
  The missingness indicator therefore carries real signal, and Part B confirms that error behaviour
  differs sharply between the two groups.
* **Frequency encoding instead of target encoding.** Encoding `publisher`/`developer` by their
  *relative frequency* uses no information about `y`, so it cannot leak the target. Target encoding
  would be stronger but would require nested cross-validation to stay honest.

In [ ]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Encode high-cardinality categoricals by their training-fold relative frequency.

    Categories never seen during `fit` are mapped to 0.0, which is the semantically
    correct signal for "this publisher/developer does not appear in the training data".
    Because the encoder is fitted inside the pipeline, it is refitted on every CV fold
    and cannot leak information across folds.
    """

    def fit(self, X: pd.DataFrame, y=None) -> 'FrequencyEncoder':
        self.feature_names_ = list(X.columns)
        self.frequency_maps_ = {col: X[col].value_counts(normalize=True) for col in X.columns}
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        return np.column_stack([
            X[col].map(self.frequency_maps_[col]).fillna(0.0).to_numpy()
            for col in self.feature_names_
        ])

    def get_feature_names_out(self, input_features=None) -> np.ndarray:
        return np.array([f'{col}_frequency' for col in self.feature_names_])


NUMERIC_FEATURES = ['critic_score', 'release_year', 'release_month',
                    'game_age_years', 'platform_count', 'has_critic_score']
ONE_HOT_FEATURES = ['genre', 'console']
FREQUENCY_FEATURES = ['publisher', 'developer', 'console']
MODEL_FEATURES = list(dict.fromkeys(NUMERIC_FEATURES + ONE_HOT_FEATURES + FREQUENCY_FEATURES))

LEAKY_COLUMNS = ['na_sales', 'jp_sales', 'pal_sales', 'other_sales']
assert not set(MODEL_FEATURES) & set(LEAKY_COLUMNS), 'Regional sales must never enter the feature set'


def build_preprocessor(scale_numeric: bool = False) -> ColumnTransformer:
    """Preprocessing block; refitted inside every CV fold, so no cross-fold information leaks."""
    numeric_steps = [('impute', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scale', StandardScaler()))
    return ColumnTransformer([
        ('numeric', Pipeline(numeric_steps), NUMERIC_FEATURES),
        ('one_hot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=30,
                                  sparse_output=False), ONE_HOT_FEATURES),
        ('frequency', FrequencyEncoder(), FREQUENCY_FEATURES),
    ])


def build_pipeline(model, scale_numeric: bool = False) -> Pipeline:
    return Pipeline([('preprocess', build_preprocessor(scale_numeric)), ('model', model)])


X = games[MODEL_FEATURES]
y_regression = games['log_total_sales'].to_numpy()
y_classification = games['is_hit'].to_numpy()

_n_encoded = build_preprocessor().fit_transform(X).shape[1]
print(f'Source feature columns : {len(MODEL_FEATURES)}')
print(f'Encoded design matrix  : {_n_encoded} columns')
print(f'Excluded as leakage    : {LEAKY_COLUMNS}')

### A.4 A single cross-validation engine

Every model in this notebook goes through the same function. It returns:

1. **out-of-fold (OOF) predictions** -- one prediction per row, always produced by a model that
   never saw that row. All error analysis in Parts B and C uses these, so the analysis reflects
   generalisation error rather than training-set memorisation;
2. **per-fold metrics** -- the spread across folds is itself a variance diagnostic.

In [ ]:
def cross_validate_oof(pipeline, X, y, cv, task='regression'):
    """Run k-fold CV once and return (out-of-fold predictions, per-fold metric frame).

    For classification the OOF vector holds P(y = 1); for regression it holds y_hat.
    A single pass is used (rather than separate predict/score passes) so each fold is fitted once.
    """
    oof = np.zeros(len(y), dtype=float)
    fold_rows = []

    for fold_index, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        fitted = pipeline.fit(X.iloc[train_idx], y[train_idx])

        if task == 'regression':
            fold_pred = fitted.predict(X.iloc[test_idx])
            fold_rows.append({
                'fold': fold_index,
                'MAE': mean_absolute_error(y[test_idx], fold_pred),
                'RMSE': mean_squared_error(y[test_idx], fold_pred) ** 0.5,
                'R2': r2_score(y[test_idx], fold_pred),
            })
        else:
            fold_pred = fitted.predict_proba(X.iloc[test_idx])[:, 1]
            fold_rows.append({
                'fold': fold_index,
                'ROC_AUC': roc_auc_score(y[test_idx], fold_pred),
                'PR_AUC': average_precision_score(y[test_idx], fold_pred),
                'MCC': matthews_corrcoef(y[test_idx], (fold_pred >= 0.5).astype(int)),
            })

        oof[test_idx] = fold_pred

    return oof, pd.DataFrame(fold_rows).set_index('fold')

### A.5 Choice of *k* -- justification

The brief requires an explicit justification of *k*, weighing dataset size, computational cost and
the bias-variance trade-off. **We use k = 5**, for four reasons:

1. **Dataset size.** With ~18.8k rows, each fold holds out ~3.8k observations. That is large enough
   for the held-out metrics -- and, importantly, for the *error distributions* this assignment
   analyses -- to be stable. The top-5% tail alone contains ~940 observations, which would be far
   noisier under a larger *k* with smaller test folds.
2. **Bias-variance of the estimator itself.** Small *k* (e.g. 2-3) trains on only 50-67% of the data
   and therefore *pessimistically biases* the performance estimate. Large *k* (e.g. LOOCV) has near-zero
   bias but high variance, because the *k* training sets overlap almost completely and their errors
   are strongly correlated. k = 5-10 is the standard compromise; at 80% training data the bias is
   already small.
3. **Class imbalance.** The positive class holds ~1,504 titles (8.0%). With **stratified** 5-fold, each
   fold still contains ~300 positives -- enough for a meaningful confusion matrix and threshold sweep.
   At k = 10 folds would carry ~150 positives each, making per-fold precision noticeably noisier.
4. **Computational cost.** Six regression models (including an $O(n^2)$ SVR) and four classifiers are
   evaluated. Moving from k = 5 to k = 10 doubles the runtime for a gain that the empirical check
   below shows to be negligible.

The check below is the empirical part of the argument: it re-runs one model at k = 3, 5 and 10 and
compares both the mean score and its spread across folds.

In [ ]:
def evaluate_k_sensitivity(candidate_ks=(3, 5, 10)) -> pd.DataFrame:
    """Empirical support for the choice of k: mean and spread of R^2 across fold counts."""
    probe_model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.08,
                                                random_state=RANDOM_STATE)
    rows = []
    for k in candidate_ks:
        cv = KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
        oof, folds = cross_validate_oof(build_pipeline(probe_model), X, y_regression, cv)
        rows.append({
            'k': k,
            'mean_fold_R2': folds['R2'].mean(),
            'std_fold_R2': folds['R2'].std(),
            'pooled_OOF_R2': r2_score(y_regression, oof),
            'pooled_OOF_RMSE': mean_squared_error(y_regression, oof) ** 0.5,
            'model_fits': k,
        })
    return pd.DataFrame(rows).set_index('k')


k_sensitivity = evaluate_k_sensitivity()
display(k_sensitivity.round(4))

**Result.** The pooled out-of-fold $R^2$ rises by about **0.019 from k = 3 to k = 5** -- the
pessimistic bias of training on only 67% of the data -- and then by a further **0.0005 from k = 5 to
k = 10**, i.e. essentially nothing for double the compute. Meanwhile the *standard deviation across
folds* grows monotonically (0.015 -> 0.018 -> 0.023): exactly the variance inflation theory predicts,
because larger *k* means smaller, noisier test folds. k = 5 is therefore the point where the bias has
been paid off but fold-level estimates are still stable. **k = 5 is used for every experiment from here on**, with
`StratifiedKFold` for the classification task so that the 8% positive rate is preserved in each fold.

In [ ]:
REGRESSION_CV = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
CLASSIFICATION_CV = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

---

## Part B -- Regression

Sections 1 and 2 of the brief are presented in reverse order: the models are fitted first
(brief section 2), because the error analysis (brief section 1) needs their out-of-fold predictions.

### B.1 Models and hyperparameters

Six models are compared. All three required families are present, plus both optional bonus models.
Every model receives **exactly the same feature set** and the same folds, so the comparison isolates
the effect of the modelling assumption rather than of the inputs.

In [ ]:
@dataclass
class ModelSpec:
    """A model together with the preprocessing variant it requires."""
    estimator: object
    scale_numeric: bool
    family: str
    hyperparameters: str


REGRESSION_MODELS = {
    'Linear Regression': ModelSpec(
        LinearRegression(), True, 'Linear (baseline)',
        'default (OLS, no regularisation); numeric features standardised'),
    'Decision Tree': ModelSpec(
        DecisionTreeRegressor(max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE),
        False, 'Single tree', 'max_depth=8, min_samples_leaf=20'),
    'Random Forest': ModelSpec(
        RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                              n_jobs=-1, random_state=RANDOM_STATE),
        False, 'Bagging ensemble', 'n_estimators=300, min_samples_leaf=2'),
    'Gradient Boosting': ModelSpec(
        HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06, random_state=RANDOM_STATE),
        False, 'Boosting ensemble', 'max_iter=400, learning_rate=0.06'),
    'KNN (bonus)': ModelSpec(
        KNeighborsRegressor(n_neighbors=15, weights='distance', n_jobs=-1),
        True, 'Instance-based', 'n_neighbors=15, weights="distance"; features standardised'),
    'SVR (bonus)': ModelSpec(
        SVR(C=1.0, epsilon=0.05, cache_size=1000), True, 'Kernel method',
        'RBF kernel, C=1.0, epsilon=0.05; features standardised'),
}

display(pd.DataFrame(
    [{'Model': name, 'Family': spec.family, 'Hyperparameters': spec.hyperparameters}
     for name, spec in REGRESSION_MODELS.items()]).set_index('Model'))

**Hyperparameter policy.** Values are lightly chosen rather than exhaustively tuned, and are stated
explicitly above. The tree depth and leaf-size constraints are deliberate: an unconstrained tree on
19k rows memorises the training fold and produces error diagnostics that describe overfitting rather
than the structure of the problem. Tuning every model by nested search would improve absolute scores
but would not change the *error structure* this assignment is about -- and the discussion in B.7
returns to what tuning could and could not fix.

In [ ]:
regression_oof = {}
regression_fold_metrics = {}
regression_summary_rows = []

for name, spec in REGRESSION_MODELS.items():
    pipeline = build_pipeline(spec.estimator, spec.scale_numeric)
    oof, folds = cross_validate_oof(pipeline, X, y_regression, REGRESSION_CV, task='regression')
    residuals = y_regression - oof

    regression_oof[name] = oof
    regression_fold_metrics[name] = folds
    regression_summary_rows.append({
        'Model': name,
        'MAE': mean_absolute_error(y_regression, oof),
        'MSE': mean_squared_error(y_regression, oof),
        'RMSE': mean_squared_error(y_regression, oof) ** 0.5,
        'R2': r2_score(y_regression, oof),
        'RMSE_fold_std': folds['RMSE'].std(),
        'R2_fold_std': folds['R2'].std(),
    })
    print(f'{name:<20s} done')

regression_summary = pd.DataFrame(regression_summary_rows).set_index('Model')
display(regression_summary.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, metric, better in zip(axes, ['RMSE', 'MAE', 'R2'], ['lower', 'lower', 'higher']):
    ordered = regression_summary[metric].sort_values(ascending=(better == 'lower'))
    colors = ['#55A868' if v == (ordered.min() if better == 'lower' else ordered.max())
              else '#4C72B0' for v in ordered]
    ax.barh(ordered.index, ordered.values, color=colors)
    ax.set_title(f'{metric} (out-of-fold, {better} is better)')
    ax.set_xlabel(metric)
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Relative performance.** The ranking is unambiguous and consistent across all three metrics:

* **Gradient Boosting** and **Random Forest** lead clearly ($R^2 \approx 0.55$ and $0.53$).
* **SVR** and **KNN** occupy the middle ($R^2 \approx 0.42$ and $0.37$) -- both capture non-linearity,
  but both are hampered by the ~100-column, largely one-hot design matrix, where Euclidean distance is
  a poor notion of similarity.
* **Linear Regression** is last ($R^2 \approx 0.32$), and the **single Decision Tree** barely improves on
  it ($0.35$). The gap between one tree and 300 of them is the largest single jump in the table --
  the signal here is not a smooth function of the features but a mass of local, interacting rules
  (this publisher on that console in that year), which averaging over many trees recovers and a single
  depth-8 tree cannot.

The `RMSE_fold_std` column is the reassurance that this ranking is real: fold-to-fold spread is roughly
an order of magnitude smaller than the gaps between models.

### B.2 Residual analysis (brief 1.1)

All diagnostics from here on use the best model, **Gradient Boosting**, on its out-of-fold
predictions. Residuals are defined as in the brief:

$$\text{Residual} = y - \hat{y}$$

on the log scale, where $y = \log(1 + \text{total\_sales})$.

In [ ]:
BEST_REGRESSOR = regression_summary['RMSE'].idxmin()
print(f'Best regression model by out-of-fold RMSE: {BEST_REGRESSOR}')

predicted = regression_oof[BEST_REGRESSOR]
residuals = y_regression - predicted
absolute_errors = np.abs(residuals)

error_frame = games.assign(
    predicted_log=predicted,
    residual=residuals,
    absolute_error=absolute_errors,
    predicted_sales=np.expm1(predicted),
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# (1) Residuals vs predicted values
axes[0, 0].scatter(predicted, residuals, s=5, alpha=0.15, color='#4C72B0')
axes[0, 0].axhline(0, color='crimson', lw=1.2)
axes[0, 0].set_title('Residuals vs predicted values')
axes[0, 0].set_xlabel('Predicted log1p(total_sales)')
axes[0, 0].set_ylabel('Residual (y - y_hat)')

# (2) Distribution of residuals
sns.histplot(residuals, bins=90, kde=True, ax=axes[0, 1], color='#DD8452')
axes[0, 1].axvline(0, color='crimson', lw=1.2, label='zero')
axes[0, 1].axvline(residuals.mean(), color='navy', ls='--', lw=1.2,
                   label=f'mean = {residuals.mean():.4f}')
axes[0, 1].axvline(np.median(residuals), color='green', ls=':', lw=1.4,
                   label=f'median = {np.median(residuals):.4f}')
axes[0, 1].set_title('Distribution of residuals')
axes[0, 1].set_xlabel('Residual')
axes[0, 1].legend(fontsize=8)

# (3) Normal Q-Q plot -- tail behaviour
stats.probplot(residuals, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Normal Q-Q plot of residuals')
axes[1, 0].get_lines()[0].set_markersize(2.5)
axes[1, 0].get_lines()[0].set_alpha(0.4)

# (4) Scale-location plot -- variance as a function of the fitted value
axes[1, 1].scatter(predicted, np.sqrt(absolute_errors), s=5, alpha=0.15, color='#937860')
binned = pd.DataFrame({'p': predicted, 's': np.sqrt(absolute_errors)})
binned['bin'] = pd.qcut(binned['p'], 25, labels=False)
trend = binned.groupby('bin').agg(p=('p', 'mean'), s=('s', 'mean'))
axes[1, 1].plot(trend['p'], trend['s'], color='crimson', lw=2, label='binned mean')
axes[1, 1].set_title('Scale-location: sqrt(|residual|) vs predicted')
axes[1, 1].set_xlabel('Predicted log1p(total_sales)')
axes[1, 1].set_ylabel('sqrt(|residual|)')
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# --- Quantitative backing for the four questions in brief 1.1 ---------------
centering_test = stats.ttest_1samp(residuals, 0.0)
heteroscedasticity_test = stats.spearmanr(absolute_errors, predicted)

print('Is the residual mean zero?')
print(f'  mean   = {residuals.mean(): .5f}   median = {np.median(residuals): .5f}')
print(f'  one-sample t-test H0: mean = 0  ->  t = {centering_test.statistic:.3f}, '
      f'p = {centering_test.pvalue:.4f}')
print()
print('Is the residual spread constant?')
print(f'  Spearman corr(|residual|, predicted) = {heteroscedasticity_test.statistic:.4f} '
      f'(p = {heteroscedasticity_test.pvalue:.2e})')
print()

decile_profile = (error_frame
    .assign(prediction_decile=pd.qcut(predicted, 10, labels=False) + 1)
    .groupby('prediction_decile')
    .agg(n=('residual', 'size'),
         mean_predicted=('predicted_log', 'mean'),
         mean_residual=('residual', 'mean'),
         residual_std=('residual', 'std'),
         MAE=('absolute_error', 'mean')))
print('Residual behaviour by decile of the predicted value:')
display(decile_profile.round(4))

#### Answers to the questions in brief 1.1

**Are the residuals centred around zero?**
On average, yes -- the mean residual is ~0.002 on the log scale and a one-sample t-test cannot reject
$H_0: \mathbb{E}[r] = 0$ (p ≈ 0.25). But the *median* residual is about **-0.026**, i.e. distinctly
negative. Mean ≈ 0 while median < 0 is the signature of a **right-skewed** error distribution: the
model slightly over-predicts the typical game, and this small, frequent positive bias is offset by a
minority of very large *under*-predictions. Global unbiasedness therefore hides a systematic pattern,
which is exactly what the rest of this section unpacks.

**Do the residuals exhibit systematic patterns?**
Yes. The decile table shows the mean residual is mildly negative through deciles 2-8 (over-prediction
of ordinary titles) and turns clearly **positive (+0.024) in the top decile** -- the model
under-predicts the games it already believes will sell well. This is textbook **regression toward the
mean**: a squared-error learner hedges towards the conditional average, and in a power-law market the
conditional average is far below the actual outcome for a blockbuster. The residuals-vs-predicted
panel shows the same thing geometrically: a hard diagonal floor on the lower-left (predictions cannot
go below zero sales, so a game with `total_sales = 0` produces exactly `residual = -prediction`), and
a plume of large positive residuals extending upward on the right.

**Is there evidence of heteroscedasticity?**
Strongly. The residual standard deviation rises monotonically from **0.055 in the lowest predicted
decile to 0.403 in the highest** -- a factor of more than seven. Spearman's correlation between
$|r|$ and $\hat{y}$ is **0.617** (p < 1e-300), and the scale-location panel rises steadily rather
than staying flat. This violates the homoskedasticity assumption of OLS, which means the linear
model's coefficient standard errors and any confidence interval derived from them are not
trustworthy. It also has a benign interpretation: `log1p` compresses the tail but does not remove
the fact that uncertainty about a title's sales grows with its commercial scale -- predicting whether
a game sells 5M or 15M is intrinsically harder than predicting whether it sells 20k or 60k.

**Is there a sector more likely to have high error rates?**
Yes -- several, and they are examined in detail in B.3. In short: the high-error sector is the
**mainstream, multi-platform, critically reviewed, Western-console blockbuster segment**, and the
low-error sector is the **niche long tail** (Visual Novel, Adventure, Strategy; handhelds like PSV,
PSP, 3DS). The mechanism is the heteroscedasticity above -- error scales with target magnitude, and
these are precisely the segments where sales are large and variable.

### B.3 Error as a function of the features (brief 1.2)

The brief asks for feature values plotted against both signed residuals and absolute errors, to
surface non-linear relationships, feature-dependent error patterns and hidden subpopulations.

In [ ]:
def plot_error_against_feature(ax_signed, ax_absolute, feature, n_bins=20):
    """Scatter + binned-mean view of residual and |error| against one numeric feature."""
    subset = error_frame[[feature, 'residual', 'absolute_error']].dropna()
    binned = subset.assign(bin=pd.qcut(subset[feature], n_bins, labels=False, duplicates='drop'))
    trend = binned.groupby('bin').agg(x=(feature, 'mean'),
                                      residual=('residual', 'mean'),
                                      absolute_error=('absolute_error', 'mean'))

    ax_signed.scatter(subset[feature], subset['residual'], s=4, alpha=0.10, color='#4C72B0')
    ax_signed.plot(trend['x'], trend['residual'], color='crimson', lw=2, marker='o', ms=3)
    ax_signed.axhline(0, color='black', lw=0.8, ls='--')
    ax_signed.set_xlabel(feature)
    ax_signed.set_ylabel('Residual')
    ax_signed.set_title(f'Residual vs {feature}')

    ax_absolute.plot(trend['x'], trend['absolute_error'], color='#C44E52', lw=2, marker='o', ms=4)
    ax_absolute.set_xlabel(feature)
    ax_absolute.set_ylabel('Mean |error|')
    ax_absolute.set_title(f'Mean absolute error vs {feature}')


numeric_features_to_probe = ['critic_score', 'release_year', 'platform_count']
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for column, feature in enumerate(numeric_features_to_probe):
    plot_error_against_feature(axes[0, column], axes[1, column], feature)
plt.tight_layout()
plt.show()

In [ ]:
def error_profile_by_category(column, min_count=100):
    """Mean |error| and mean signed residual per category level."""
    return (error_frame
            .groupby(column)
            .agg(n=('absolute_error', 'size'),
                 MAE=('absolute_error', 'mean'),
                 mean_residual=('residual', 'mean'),
                 mean_actual_sales=('total_sales', 'mean'))
            .query('n >= @min_count')
            .sort_values('MAE', ascending=False))


genre_errors = error_profile_by_category('genre')
console_errors = error_profile_by_category('console')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=genre_errors['MAE'], y=genre_errors.index, ax=axes[0], color='#4C72B0')
axes[0].axvline(absolute_errors.mean(), color='crimson', ls='--',
                label=f'overall MAE = {absolute_errors.mean():.4f}')
axes[0].set_title('Mean absolute error by genre (n >= 100)')
axes[0].set_xlabel('MAE (log scale units)')
axes[0].legend(fontsize=8)

sns.barplot(x=console_errors['MAE'], y=console_errors.index, ax=axes[1], color='#DD8452')
axes[1].axvline(absolute_errors.mean(), color='crimson', ls='--')
axes[1].set_title('Mean absolute error by console (n >= 100)')
axes[1].set_xlabel('MAE (log scale units)')

plt.tight_layout()
plt.show()

display(genre_errors.round(4))

In [ ]:
# Two hidden subpopulations surfaced by the plots above
critic_split = (error_frame.groupby('has_critic_score')
                .agg(n=('absolute_error', 'size'),
                     MAE=('absolute_error', 'mean'),
                     mean_actual_sales=('total_sales', 'mean'),
                     hit_rate=('is_hit', 'mean')))
critic_split.index = ['No critic score', 'Has critic score']

platform_split = (error_frame
                  .assign(platforms=error_frame['platform_count'].clip(upper=6))
                  .groupby('platforms')
                  .agg(n=('absolute_error', 'size'),
                       MAE=('absolute_error', 'mean'),
                       mean_actual_sales=('total_sales', 'mean')))

era_split = (error_frame
             .assign(era=pd.cut(error_frame['release_year'],
                                [1976, 1995, 2000, 2005, 2010, 2015, 2021]))
             .groupby('era', observed=True)
             .agg(n=('absolute_error', 'size'),
                  MAE=('absolute_error', 'mean'),
                  mean_actual_sales=('total_sales', 'mean')))

print('Subpopulation 1 -- critic-score availability:'); display(critic_split.round(4))
print('Subpopulation 2 -- number of platforms the title shipped on (6 = "6 or more"):')
display(platform_split.round(4))
print('Subpopulation 3 -- release era:'); display(era_split.round(4))

#### What the feature-error plots reveal

**Non-linear, non-monotonic error in `critic_score`.** Absolute error is low for poorly reviewed
titles, rises steeply above a score of roughly 7, and is highest at the very top of the scale. The
signed-residual trend crosses zero and turns positive in the same region. Critic score is a *weak*
predictor of sales at the low end (bad games reliably sell little) but a *necessary-not-sufficient*
one at the high end -- a 9.0 game may be a 10M-unit phenomenon or a critically adored commercial
failure, and the model cannot tell the two apart. HW1 already found this relationship to be weak and
outlier-driven; here the same fact shows up as a feature-dependent error pattern.

**Monotone error in `platform_count`.** MAE climbs from **0.109** for single-platform titles to
**0.210** for titles released on six or more platforms -- a near-doubling. Multi-platform release is
a proxy for publisher confidence and budget, so this subpopulation is precisely the high-sales,
high-variance segment; it is also the segment where one franchise entry can behave completely
differently from the next.

**Non-monotone error in `release_year`.** Error is high for the pre-2000 era (MAE ~0.18), falls to a
minimum in 2005-2010 (~0.12), and stays low thereafter (~0.11 for 2015-2020). Two forces overlap: the
early era is thinly sampled *and* survivorship-biased (only commercially notable retro titles were
recorded at all, so the recorded ones are unusually large), while the 2005-2010 peak of the DS/Wii/PS2
era gives the model an enormous, homogeneous training population to learn from.

**Hidden subpopulation: critic-score availability.** Titles *with* a critic score have MAE **0.203**
against **0.115** for titles without -- a 76% higher error, on a group that is only 22% of the data.
This looks paradoxical (more information, worse predictions) but it is not: `has_critic_score` is a
proxy for "commercially significant enough to be reviewed". Mean actual sales in that group are far
higher, and by the heteroscedasticity result, higher sales mean larger errors. The extra feature does
not cause the error; it *selects* the hard subpopulation.

**Sector answer.** Combining the panels: the high-error sector is **Shooter / Action-Adventure /
Music / Fighting / Sports titles, on Western home consoles (PS, X360, PS2, PS3, PS4), released on
many platforms, with a critic score**. The low-error sector is **Visual Novel / Adventure / Strategy
on handhelds and PC**. This is not a set of unrelated findings -- it is one finding seen from several
angles: *error magnitude tracks commercial scale and its variance*.

### B.4 Analysis of extreme errors (brief 1.3)

The top 5% of absolute errors are isolated and examined individually, then attributed to one of the
three causes named in the brief: data quality, model limitation, or genuinely rare cases.

In [ ]:
EXTREME_ERROR_QUANTILE = 0.95
extreme_cutoff = np.quantile(absolute_errors, EXTREME_ERROR_QUANTILE)
extreme_errors = error_frame[error_frame['absolute_error'] >= extreme_cutoff].copy()
extreme_errors['direction'] = np.where(extreme_errors['residual'] > 0,
                                       'under-predicted', 'over-predicted')

print(f'Top {1 - EXTREME_ERROR_QUANTILE:.0%} of absolute errors: {len(extreme_errors):,} rows '
      f'(cutoff |residual| = {extreme_cutoff:.4f})')
print(extreme_errors['direction'].value_counts().to_string())
print()

comparison = pd.DataFrame({
    'Top 5% errors': {
        'n': len(extreme_errors),
        'mean total_sales (M)': extreme_errors['total_sales'].mean(),
        'median total_sales (M)': extreme_errors['total_sales'].median(),
        'share with sales = 0': (extreme_errors['total_sales'] == 0).mean(),
        'share missing critic_score': extreme_errors['critic_score'].isna().mean(),
        'mean platform_count': extreme_errors['platform_count'].mean(),
        'hit rate': extreme_errors['is_hit'].mean(),
    },
    'All rows': {
        'n': len(error_frame),
        'mean total_sales (M)': error_frame['total_sales'].mean(),
        'median total_sales (M)': error_frame['total_sales'].median(),
        'share with sales = 0': (error_frame['total_sales'] == 0).mean(),
        'share missing critic_score': error_frame['critic_score'].isna().mean(),
        'mean platform_count': error_frame['platform_count'].mean(),
        'hit rate': error_frame['is_hit'].mean(),
    },
})
display(comparison.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(extreme_errors['residual'], bins=50, ax=axes[0], color='#C44E52')
axes[0].axvline(0, color='black', lw=1)
axes[0].set_title('Signed residual within the top 5% of |errors|')
axes[0].set_xlabel('Residual')

genre_share = pd.DataFrame({
    'Top 5% errors': extreme_errors['genre'].value_counts(normalize=True),
    'All rows': error_frame['genre'].value_counts(normalize=True),
}).dropna().sort_values('Top 5% errors', ascending=False).head(10)
genre_share.plot.barh(ax=axes[1], width=0.8)
axes[1].invert_yaxis()
axes[1].set_title('Genre composition: extreme errors vs overall')
axes[1].set_xlabel('Share of rows')
axes[1].legend(fontsize=8)

axes[2].scatter(error_frame['predicted_sales'], error_frame['total_sales'],
                s=4, alpha=0.10, color='#4C72B0', label='all rows')
axes[2].scatter(extreme_errors['predicted_sales'], extreme_errors['total_sales'],
                s=6, alpha=0.45, color='#C44E52', label='top 5% |error|')
limit = [0.005, 25]
axes[2].plot(limit, limit, color='black', lw=1, ls='--', label='perfect prediction')
axes[2].set_xscale('log'); axes[2].set_yscale('log')
axes[2].set_xlim(0.005, 25); axes[2].set_ylim(0.005, 25)
axes[2].set_xlabel('Predicted sales (M, back-transformed)')
axes[2].set_ylabel('Actual sales (M)')
axes[2].set_title('Predicted vs actual (log-log)')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
display_columns = ['title', 'console', 'genre', 'publisher', 'critic_score',
                   'release_year', 'platform_count', 'total_sales', 'predicted_sales',
                   'absolute_error']

print('=== The 12 largest absolute errors ===')
display(extreme_errors.nlargest(12, 'absolute_error')[display_columns].round(3)
        .reset_index(drop=True))

print('=== Largest UNDER-predictions: blockbusters the model did not see coming ===')
display(extreme_errors.nlargest(10, 'residual')[display_columns].round(3).reset_index(drop=True))

print('=== Largest OVER-predictions: titles the model expected to sell and they did not ===')
display(extreme_errors.nsmallest(10, 'residual')[display_columns].round(3).reset_index(drop=True))

#### Where do the extreme errors come from?

**The tail is overwhelmingly one-sided.** Of the 942 extreme errors, **760 (81%) are
under-predictions** and only 182 are over-predictions. Their mean actual sales are **2.21M against
0.35M** across the frame, and their hit rate is far above the base rate. The model's worst failures
are not random noise -- they are concentrated in the blockbuster tail, in one direction.

Examining the individual rows, three distinct causes appear, matching the brief's three categories:

**1. Data quality issues.** The over-prediction list is dominated by rows that are not really
"a game's lifetime sales" at all:

* `Resident Evil 5: Gold Edition`, `LittleBigPlanet: Game of the Year Edition` -- re-releases with
  recorded `total_sales = 0.00`, while the model sees a strong franchise, a good console and a major
  publisher and predicts ~1.9M. **1.2% of extreme errors have `total_sales = 0`**, and these are
  arithmetically the largest possible over-prediction.
* `Undertale (PS4)`: recorded 0.01M -- a real commercial phenomenon whose sales were overwhelmingly
  digital, and VGChartz tracks physical retail. The recorded value is not wrong, it just measures
  something different from what the label implies.
* Rows whose title *itself* leaks the problem, e.g. `Pokémon Mystery Dungeon: Red Rescue Team
  (US weekly sales)` -- a partial-period figure sitting in a lifetime-sales column.

These are not modelling failures. They are the target variable being **inconsistently defined**
across rows, exactly the "data is a product of a measurement process" point from HW1.

**2. Model limitations.** The under-prediction list is the clearer failure of the model itself.
`Grand Theft Auto V (X360)` sold 15.86M and was predicted at 0.61M; `Halo 4` sold 9.96M, predicted
0.37M. The features simply do not contain what makes these titles different: there is no *franchise
identity*, no marketing spend, no pre-order volume, no sequel-number, no review aggregate for those
particular rows (**55% of extreme errors have no `critic_score`**, and GTA V on X360 is one of them).
The model knows "Rockstar Games, Action, 2013, X360" -- and the median such game sells a few hundred
thousand copies. Under a squared-error loss on a log target, predicting the conditional median is the
rational response, so the model is behaving correctly given an inadequate feature set.

**3. Rare / exceptional cases.** A residual class is irreducible. `Microsoft Flight Simulator (1996)`
and `Warzone 2100 (PS, 1999)` are pre-2000 titles from a thin, survivorship-biased region of the data.
Cultural breakout hits (`Just Dance 4`, `Cooking Mama`) created entire casual-gaming categories rather
than fitting an existing one. No feature available *before release* would have flagged them, because
the information that made them succeed did not exist at prediction time. This is aleatoric uncertainty
in a hit-driven market, not a fixable defect.

**Practical consequence.** Roughly the first category should be *cleaned* (filter zero-sales rows,
flag edition/re-release rows via the title string, drop partial-period rows), the second should be
*engineered away* (franchise identifier, sequel index, first-party publisher flag), and only the third
should be accepted as a floor on achievable accuracy.

### B.5 Statistical properties of the errors (brief 1.4)

MAE, residual standard deviation, skewness and kurtosis are computed for every model, so the
distributional properties can be compared across modelling assumptions rather than read off a
single model.

In [ ]:
error_statistics = []
for name, oof in regression_oof.items():
    model_residuals = y_regression - oof
    error_statistics.append({
        'Model': name,
        'MAE': mean_absolute_error(y_regression, oof),
        'Residual mean': model_residuals.mean(),
        'Residual median': np.median(model_residuals),
        'Residual std': model_residuals.std(),
        'Skewness': stats.skew(model_residuals),
        'Excess kurtosis': stats.kurtosis(model_residuals, fisher=True),
        'P95 |error|': np.quantile(np.abs(model_residuals), 0.95),
        'Max |error|': np.abs(model_residuals).max(),
    })

error_statistics = pd.DataFrame(error_statistics).set_index('Model')
display(error_statistics.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for name, oof in regression_oof.items():
    sns.kdeplot(y_regression - oof, ax=axes[0], label=name, lw=1.5, clip=(-1.2, 1.6))
axes[0].axvline(0, color='black', lw=0.8, ls='--')
axes[0].set_title('Residual densities across models')
axes[0].set_xlabel('Residual')
axes[0].legend(fontsize=8)

skew_kurt = error_statistics[['Skewness', 'Excess kurtosis']]
skew_kurt.plot.barh(ax=axes[1], width=0.8)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title('Shape of the residual distribution (0 = Gaussian)')
axes[1].set_xlabel('Value')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

#### Interpretation

**Skewness -- systematic bias.** Every model produces a **positively skewed** residual distribution
(Gradient Boosting ~1.76, Linear Regression ~2.21, Gaussian = 0). Positive skew means the long tail
is on the *under-prediction* side, confirming the finding from B.2 and B.4: models are systematically
conservative about blockbusters. Note the ordering -- the linear model is the *most* skewed and the
boosted model the least, so flexible models partially, but only partially, correct the bias.

**Kurtosis -- instability and heavy tails.** Excess kurtosis is **~8.6 for Gradient Boosting** and
**~9.0-9.8 for the others**, against 0 for a normal distribution. Errors are far more concentrated
near zero *and* far more prone to extreme values than a Gaussian. Practically this means:

* the RMSE is dominated by a small minority of rows -- the P95 absolute error (0.453) is **3.4x the
  MAE** (0.134), and the maximum (2.35) is a further **5x** beyond the P95;
* any prediction interval built on a normal assumption would be badly mis-calibrated -- too wide for
  the typical game, catastrophically too narrow for the tail;
* re-fitting on a slightly different sample can move the tail noticeably, which is the *instability*
  the brief refers to. It is visible in the `RMSE_fold_std` column of B.1, where the boosted model
  has slightly *higher* fold-to-fold variance than the forest despite better mean performance.

**MAE vs RMSE.** Across all six models MAE is far below RMSE (e.g. 0.134 vs 0.212 for the best
model). Under a Gaussian the ratio would be about 0.8; here it is around 0.63. That gap is itself a
measure of tail-dominance, and it is why MAE is reported alongside RMSE throughout: MAE describes the
typical game, RMSE describes the exposure to the tail, and in this dataset they are not
interchangeable.

**A note on SVR.** The SVR is the only model with a materially non-zero mean residual (+0.041): its
$\varepsilon$-insensitive loss ignores errors under 0.05, which combined with the asymmetric target
distribution leaves a small persistent negative bias. It is a clean illustration of how the *loss
function*, not just the model class, shapes the error distribution.

### B.6 Are the linear-model assumptions satisfied?

Before the discussion, the OLS assumptions are tested explicitly rather than asserted.

In [ ]:
linear_residuals = y_regression - regression_oof['Linear Regression']

assumption_checks = pd.DataFrame([
    {'Assumption': 'Zero-mean errors',
     'Test': 'one-sample t-test on residual mean',
     'Statistic': f'p = {stats.ttest_1samp(linear_residuals, 0).pvalue:.3f}',
     'Verdict': 'Satisfied on average'},
    {'Assumption': 'Homoskedasticity',
     'Test': 'Spearman corr(|residual|, fitted)',
     'Statistic': f'rho = {stats.spearmanr(np.abs(linear_residuals), regression_oof["Linear Regression"]).statistic:.3f}',
     'Verdict': 'Clearly violated'},
    {'Assumption': 'Normality of errors',
     'Test': "D'Agostino-Pearson normality test",
     'Statistic': f'p = {stats.normaltest(linear_residuals).pvalue:.2e}',
     'Verdict': 'Clearly violated (skew + heavy tails)'},
    {'Assumption': 'Linearity in the features',
     'Test': 'R2 gain of boosted trees over OLS',
     'Statistic': (f'{regression_summary.loc["Gradient Boosting", "R2"]:.3f} vs '
                   f'{regression_summary.loc["Linear Regression", "R2"]:.3f}'),
     'Verdict': 'Violated: large non-linear/interaction signal remains'},
    {'Assumption': 'Independent observations',
     'Test': 'structural check on the data',
     'Statistic': 'same title on multiple consoles / franchise entries',
     'Verdict': 'Violated: rows are clustered by title and franchise'},
]).set_index('Assumption')

display(assumption_checks)

The independence violation deserves emphasis because it is invisible to any residual plot. Rows are
**not** independent draws: the same title appears on up to six consoles, and franchise entries share
a brand, an audience and a marketing machine. Random k-fold splitting therefore places `GTA V (PS3)`
in the training fold while `GTA V (X360)` is being predicted, which makes every score in this
notebook **mildly optimistic**. A grouped split (`GroupKFold` on `title`, or on a franchise key) would
give a more honest estimate of performance on a genuinely new title, and is the single most important
methodological improvement recommended in Part D.

### B.7 Regression discussion (brief section 2)

**Relative performance across metrics.** The ranking is stable across MSE, RMSE, MAE and $R^2$:
Gradient Boosting ($R^2 \approx 0.55$) ≳ Random Forest ($0.53$) > SVR ($0.42$) > KNN ($0.37$) >
Decision Tree ($0.35$) > Linear Regression ($0.32$). Two observations matter more than the ordering.
First, the metrics *disagree in emphasis*: Random Forest and Gradient Boosting are nearly tied on MAE
(0.135 vs 0.134) but separate more on RMSE, meaning the boosted model's advantage lies mainly in
handling the tail, not the typical game. Second, no model exceeds $R^2 = 0.55$; roughly **45% of the
variance in log-sales is not explained by any of them**, which points at the feature set rather than
at the algorithms.

**Model assumptions vs reality.** Section B.6 shows OLS is operating outside its assumptions on
three counts (heteroscedasticity, non-normality, non-linearity) and that all models violate
independence. The consequences are visible and specific: OLS coefficient standard errors are
unreliable, its residuals are the most skewed of any model, and it is the only family that cannot
represent the interaction structure the trees exploit.

**Bias-variance trade-off.** The six models trace the curve cleanly. Linear Regression is the
**high-bias, low-variance** end -- it underfits badly (worst $R^2$) but has the tightest fold-to-fold
spread. A single depth-8 tree lowers bias only slightly. Random Forest attacks *variance* by averaging
300 decorrelated trees, and delivers the largest single jump in the table (+0.18 $R^2$ over one tree).
Gradient Boosting attacks *bias* by fitting residuals sequentially, gains a further +0.02 $R^2$, but
pays for it with a slightly larger fold-to-fold RMSE spread -- more capacity, more sensitivity to the
sample. KNN sits at high variance with no compensating structure: in a ~100-dimensional, mostly
one-hot space, the 15 nearest neighbours of a game are not meaningfully similar to it.

**Sensitivity to outliers and noise.** The `Max |error|` and `P95 |error|` columns in B.5 quantify
this. Linear Regression is the most exposed -- a squared loss on a heavy-tailed target lets a handful
of blockbusters pull the entire fitted hyperplane, which is why its residuals are the most skewed.
Tree ensembles are structurally more robust because splits depend on the *ordering* of feature
values, not their magnitude, so an extreme target value can only distort the leaf it lands in. SVR is
robust by construction ($\varepsilon$-insensitive loss ignores small errors, and the hinge bounds the
influence of large ones), which is why it beats KNN despite the unfavourable geometry -- but that same
loss leaves it with a persistent bias.

**Where linear models fail, and why.** Three concrete failures, all diagnosed above: (a) the response
is *interaction-driven* -- the effect of `console` depends on `genre`, `release_year` and
`publisher` jointly, and OLS can only add main effects; (b) the effect of `critic_score` is
**non-monotone in its error behaviour**, essentially flat below 7 and explosive above it, which a
single slope cannot express; (c) the error variance grows by 7× across the prediction range, so the
one assumption OLS most depends on for inference is the one most badly violated.

**Where tree-based models win.** Precisely where the structure is *conditional and local*.
Frequency-encoded `publisher`/`developer` become useful only when a tree can split on them and then
ask a different question in each branch. Trees also absorb the median-imputed `critic_score` and the
`has_critic_score` flag gracefully: they can isolate "no critic score" as its own region rather than
being forced to treat an imputed 7.4 as a real measurement.

**Interpretability vs predictive performance.** The trade-off is real but small in absolute terms.
Linear Regression yields directly readable coefficients and costs 0.23 $R^2$. A depth-8 tree is
still human-readable end-to-end and recovers only 0.03 of that. The full 0.23 requires 300 trees or
400 boosting iterations, which are not inspectable as rules -- though they remain *explainable* post
hoc via permutation importance or SHAP. For this particular problem the honest summary is that a
transparent model would sacrifice about 40% of the achievable explained variance.

**Preferred model.** **Gradient Boosting (`HistGradientBoostingRegressor`)**, for four reasons:
best RMSE, MSE and $R^2$; the *least* skewed and least heavy-tailed residual distribution of the six,
meaning its errors are the closest to well-behaved; native handling of the missing/imputed and
mixed-type features without extra preprocessing; and training in seconds, which keeps the
error-analysis loop of this assignment fast. Random Forest is the reasonable alternative -- it is
statistically indistinguishable on MAE and slightly more stable across folds -- and would be the
choice if fold-to-fold consistency mattered more than tail accuracy.

**What the modelling stage taught.** Three things that were not obvious in advance. First, that the
*target definition* was the highest-leverage decision in the whole notebook: excluding the regional
sales columns took $R^2$ from a meaningless ~1.0 to an honest 0.55, and the log transform changed
which errors the loss function even cared about. Second, that a better metric is not the same as a
better *error structure* -- Gradient Boosting and Random Forest are nearly tied on MAE while
differing meaningfully in tail behaviour, and only the error analysis reveals that. Third, that the
performance ceiling here is set by **missing features, not by model capacity**: every model, from
OLS to boosting, fails on the same rows in the same direction, which is the signature of a feature
set that lacks the relevant information rather than of an algorithm that cannot use it.

---

## Part C -- Classification Error Analysis (brief section 3)

### C.1 Problem definition

A title is labelled a **hit** if it sold at least **1.0M units**. The threshold is not arbitrary:
"million-seller" is the industry's own standard for commercial success, and it produces a
**7.99% positive rate** -- imbalanced enough to make accuracy useless and the threshold analysis of
section C.5 genuinely necessary, but with ~1,500 positives, still enough to estimate precision and
recall reliably.

The feature set, preprocessing and fold seed are **identical to Part B**; only the target and the
estimator family change. `StratifiedKFold` preserves the 8% positive rate in every fold.

In [ ]:
CLASSIFICATION_MODELS = {
    'Logistic Regression': ModelSpec(
        LogisticRegression(max_iter=3000), True, 'Linear (baseline)',
        'default L2, max_iter=3000; features standardised'),
    'Decision Tree': ModelSpec(
        DecisionTreeClassifier(max_depth=8, min_samples_leaf=25, random_state=RANDOM_STATE),
        False, 'Single tree', 'max_depth=8, min_samples_leaf=25'),
    'Random Forest': ModelSpec(
        RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                               n_jobs=-1, random_state=RANDOM_STATE),
        False, 'Bagging ensemble', 'n_estimators=300, min_samples_leaf=2'),
    'Gradient Boosting': ModelSpec(
        HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06,
                                       random_state=RANDOM_STATE),
        False, 'Boosting ensemble', 'max_iter=400, learning_rate=0.06'),
}

classification_oof = {}
classification_rows = []

for name, spec in CLASSIFICATION_MODELS.items():
    pipeline = build_pipeline(spec.estimator, spec.scale_numeric)
    probabilities, folds = cross_validate_oof(pipeline, X, y_classification,
                                              CLASSIFICATION_CV, task='classification')
    labels = (probabilities >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_classification, labels).ravel()

    classification_oof[name] = probabilities
    classification_rows.append({
        'Model': name,
        'ROC_AUC': roc_auc_score(y_classification, probabilities),
        'PR_AUC': average_precision_score(y_classification, probabilities),
        'Precision': precision_score(y_classification, labels, zero_division=0),
        'Recall': recall_score(y_classification, labels),
        'F1': f1_score(y_classification, labels),
        'MCC': matthews_corrcoef(y_classification, labels),
        'Balanced acc.': balanced_accuracy_score(y_classification, labels),
        'Brier': brier_score_loss(y_classification, probabilities),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
    })
    print(f'{name:<20s} done')

classification_summary = pd.DataFrame(classification_rows).set_index('Model')
print(f'\nPositive class: {y_classification.sum():,} of {len(y_classification):,} '
      f'({y_classification.mean():.2%}) -- a model predicting "never a hit" would score '
      f'{1 - y_classification.mean():.2%} accuracy.')
display(classification_summary.round(4))

**Reading the table.** Gradient Boosting and Random Forest both reach **ROC-AUC ≈ 0.90**, which sounds
excellent -- but their **PR-AUC is only ~0.56**, and at the default 0.5 threshold recall is
**0.21-0.32**. This gap is the most important single fact in Part C: ROC-AUC is computed against the
majority class and is flattered by 17,325 easy negatives, whereas PR-AUC and MCC reflect performance
on the 8% that matter. **Accuracy is not reported at all**: predicting "never a hit" would score
92.0% and be completely useless. Gradient Boosting is selected as the primary classifier -- highest
PR-AUC, F1, MCC and the lowest Brier score.

### C.2 Confusion matrix analysis (brief 3.1)

In [ ]:
BEST_CLASSIFIER = classification_summary['PR_AUC'].idxmax()
print(f'Primary classifier (highest PR-AUC): {BEST_CLASSIFIER}')

hit_probability = classification_oof[BEST_CLASSIFIER]
predicted_hit = (hit_probability >= 0.5).astype(int)
is_correct = predicted_hit == y_classification

classification_frame = games.assign(
    hit_probability=hit_probability,
    predicted_hit=predicted_hit,
    is_correct=is_correct,
    outcome=np.select(
        [(y_classification == 1) & (predicted_hit == 1),
         (y_classification == 0) & (predicted_hit == 0),
         (y_classification == 0) & (predicted_hit == 1)],
        ['True Positive', 'True Negative', 'False Positive'],
        default='False Negative'),
)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for ax, (name, probabilities) in zip(axes, classification_oof.items()):
    matrix = confusion_matrix(y_classification, (probabilities >= 0.5).astype(int))
    normalised = matrix / matrix.sum(axis=1, keepdims=True)
    annotations = np.array([[f'{matrix[i, j]:,}\n({normalised[i, j]:.1%})' for j in range(2)]
                            for i in range(2)])
    sns.heatmap(normalised, annot=annotations, fmt='', cmap='Blues', cbar=False, ax=ax,
                vmin=0, vmax=1, xticklabels=['Pred: not hit', 'Pred: hit'],
                yticklabels=['True: not hit', 'True: hit'], annot_kws={'size': 9})
    ax.set_title(f'{name}\n(threshold = 0.5, row-normalised)', fontsize=9)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = confusion_matrix(y_classification, predicted_hit).ravel()
print(f'{BEST_CLASSIFIER} at threshold 0.5')
print(f'  True Negatives : {tn:6,}   False Positives: {fp:6,}')
print(f'  False Negatives: {fn:6,}   True Positives : {tp:6,}')
print(f'  Precision {tp / (tp + fp):.3f} | Recall {tp / (tp + fn):.3f} | '
      f'FN:FP ratio {fn / max(fp, 1):.1f} : 1')
print(f'  Missed sales volume: {games.loc[(y_classification == 1) & (predicted_hit == 0), "total_sales"].sum():,.0f}M units '
      f'of the {games.loc[y_classification == 1, "total_sales"].sum():,.0f}M units in the hit class')

#### Interpretation

**The matrix is dominated by one cell.** At the default threshold the model produces
**1,017 false negatives against 191 false positives** -- a 5.3 : 1 imbalance. Precision is high
(0.72) and recall is low (0.32): when the model says "hit" it is usually right, but it stays silent
for two thirds of the actual hits. This is the expected behaviour of a probabilistic classifier
trained on an 8% positive rate and then thresholded at 0.5: the decision rule "P(hit) > 0.5" demands
that a title be *more likely than not* to be a million-seller, which is a very strong claim when the
prior is 8%. The model is not broken -- **the threshold is mis-specified for the class balance**,
which is exactly what section C.5 corrects.

**Which error is more critical here?** It depends on the decision the model supports, and the two
realistic use-cases point in opposite directions:

* **Investment / greenlighting** (which titles to fund or manufacture inventory for): a **false
  positive** is expensive -- money spent on a game that does not sell, plus unsold physical stock.
  Precision matters, and the default threshold is defensible.
* **Discovery / portfolio screening** (which titles deserve a closer look): a **false negative** is
  the costly error -- a missed million-seller is missed revenue and, in a hit-driven market where the
  top few percent of titles carry most of the value, a single miss can outweigh many wasted
  evaluations. Recall matters, and the threshold should drop.

The second framing is the more natural one for this dataset, and the numbers back it: the 1,017
false negatives account for **1,805M of the 3,364M units (54%)** sold by the hit class -- the model
overlooks more than half the commercial value it was built to find. Because the two framings genuinely
conflict, the responsible output is not one confusion matrix but the **threshold curve** of C.5, from
which each stakeholder can pick their own operating point.

**Where does the model fail systematically?** The differences between the four matrices are
informative. Logistic Regression and the single tree find almost no positives at all at 0.5
(recall ≈ 0.20); the ensembles roughly double that. But all four fail in the *same direction* on the
*same kind of row* -- the failure is a property of the problem, not of any one algorithm. Section
C.4 localises exactly which rows.

### C.3 Probability-based analysis (brief 3.2)

The objective is to identify **high-confidence errors** -- cases where the model was not merely wrong
but confidently wrong, which are the dangerous ones in deployment.

In [ ]:
confidence_in_prediction = np.maximum(hit_probability, 1 - hit_probability)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.kdeplot(x=hit_probability[is_correct], ax=axes[0], label='correct', fill=True,
            clip=(0, 1), lw=1.5)
sns.kdeplot(x=hit_probability[~is_correct], ax=axes[0], label='incorrect', fill=True,
            clip=(0, 1), lw=1.5)
axes[0].axvline(0.5, color='black', ls='--', lw=1)
axes[0].set_yscale('log')
axes[0].set_xlabel('Predicted P(hit)')
axes[0].set_title('Predicted probability: correct vs incorrect')
axes[0].legend(fontsize=8)

sns.boxplot(x=classification_frame['outcome'], y=confidence_in_prediction, ax=axes[1],
            order=['True Negative', 'True Positive', 'False Positive', 'False Negative'])
axes[1].set_ylabel('Confidence in the predicted class')
axes[1].set_xlabel('')
axes[1].set_title('Confidence by confusion-matrix cell')
axes[1].tick_params(axis='x', rotation=20)

fraction_positive, mean_predicted = calibration_curve(y_classification, hit_probability,
                                                      n_bins=12, strategy='quantile')
axes[2].plot(mean_predicted, fraction_positive, marker='o', lw=1.8, label=BEST_CLASSIFIER)
axes[2].plot([0, 1], [0, 1], ls='--', color='black', lw=1, label='perfect calibration')
axes[2].set_xlabel('Mean predicted P(hit)')
axes[2].set_ylabel('Observed hit rate')
axes[2].set_title(f'Calibration curve (Brier = '
                  f'{brier_score_loss(y_classification, hit_probability):.4f})')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
HIGH_CONFIDENCE = 0.80
high_confidence_fp = (hit_probability >= HIGH_CONFIDENCE) & (y_classification == 0)
high_confidence_fn = (hit_probability <= 1 - HIGH_CONFIDENCE) & (y_classification == 1)

print(f'Mean P(hit)                     : correct {hit_probability[is_correct].mean():.4f} | '
      f'incorrect {hit_probability[~is_correct].mean():.4f}')
print(f'Mean confidence in predicted class: correct {confidence_in_prediction[is_correct].mean():.4f} | '
      f'incorrect {confidence_in_prediction[~is_correct].mean():.4f}')
print()
print(f'High-confidence false positives (P >= {HIGH_CONFIDENCE:.2f}, not a hit): '
      f'{high_confidence_fp.sum():,}  ({high_confidence_fp.sum() / max(fp, 1):.1%} of all FP)')
print(f'High-confidence false negatives (P <= {1 - HIGH_CONFIDENCE:.2f}, actually a hit): '
      f'{high_confidence_fn.sum():,}  ({high_confidence_fn.sum() / max(fn, 1):.1%} of all FN)')

probe_columns = ['title', 'console', 'genre', 'publisher', 'critic_score',
                 'release_year', 'platform_count', 'total_sales', 'hit_probability']

print('\n=== Most confident FALSE POSITIVES: predicted hits that were not ===')
display(classification_frame[high_confidence_fp].nlargest(8, 'hit_probability')[probe_columns]
        .round(3).reset_index(drop=True))

print('=== Most confident FALSE NEGATIVES: hits the model was sure would flop ===')
display(classification_frame[high_confidence_fn].nsmallest(8, 'hit_probability')[probe_columns]
        .round(3).reset_index(drop=True))

#### Interpretation

**Confidence does separate right from wrong -- but weakly, and asymmetrically.** Mean confidence in
the predicted class is **0.95 when correct and 0.78 when incorrect**, so the model's own uncertainty
is a usable signal: routing low-confidence predictions to a human reviewer would catch a
disproportionate share of the errors. The calibration curve is close to the diagonal (Brier ≈ 0.050),
so the probabilities can be read as approximately real probabilities rather than arbitrary scores.

**But the two error types behave completely differently.** Only **13 false positives (6.8%)** are
high-confidence, while **589 false negatives (58%)** are -- the model assigns P(hit) ≤ 0.20 to more
than half the hits it misses. The box plot shows this directly: the False Positive box sits low
(hesitant errors), the False Negative box sits high (confident errors). **The model is not uncertain
about the hits it misses; it is confidently wrong about them.**

**Why.** Look at the individual rows. `No Man's Sky` (2.26M actual, P = 0.002) is a game whose sales
were driven by a marketing phenomenon no feature captures. `Dragon Quest XI (3DS)`,
`Tomodachi Collection: New Life (3DS)` and `Pokémon Conquest (DS)` are **Japan-dominated titles** --
HW1's second core insight was precisely that Japan behaves as a decoupled market
($r \approx 0.07\text{–}0.21$ with Western regions). The model has learned a Western-console,
Western-publisher notion of what a hit looks like, so a Japanese handheld million-seller is not a
borderline case to it; it is confidently outside the learned pattern. `The Elder Scrolls Online`
(an MMO) and `Minecraft: Story Mode` (an episodic release) are similar: business models the training
data barely contains.

**The practical warning.** A confidence threshold is a *safety net for the model's known unknowns*,
and it works here for false positives -- but it would not catch the majority of the misses, because
those come from regions of the feature space where the model is wrong *and* certain. That is a
coverage failure, and coverage failures are fixed with features and data, not with thresholds.

### C.4 Error as a function of the features (brief 3.3)

Feature distributions are compared between correctly and incorrectly classified rows, to locate the
regions of feature space that are prone to misclassification.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
correctness_label = np.where(is_correct, 'Correct', 'Incorrect')

for ax, feature in zip(axes[:3], ['critic_score', 'release_year', 'platform_count']):
    sns.violinplot(x=correctness_label, y=classification_frame[feature], ax=ax,
                   order=['Correct', 'Incorrect'], cut=0, inner='quartile')
    subset_correct = classification_frame.loc[is_correct, feature].dropna()
    subset_incorrect = classification_frame.loc[~is_correct, feature].dropna()
    p_value = stats.mannwhitneyu(subset_correct, subset_incorrect).pvalue
    ax.set_title(f'{feature}\nMann-Whitney p = {p_value:.1e}', fontsize=9)
    ax.set_xlabel('')

critic_availability = (classification_frame
                       .assign(has_score=np.where(classification_frame['has_critic_score'] == 1,
                                                  'Has critic score', 'No critic score'))
                       .groupby('has_score')['is_correct']
                       .apply(lambda s: 1 - s.mean()))
sns.barplot(x=critic_availability.index, y=critic_availability.values, ax=axes[3], color='#C44E52')
axes[3].set_ylabel('Error rate')
axes[3].set_xlabel('')
axes[3].set_title('Error rate by critic-score availability', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
def misclassification_profile(column, min_count=150):
    return (classification_frame
            .assign(is_error=~classification_frame['is_correct'])
            .groupby(column)
            .agg(n=('is_error', 'size'),
                 error_rate=('is_error', 'mean'),
                 hit_rate=('is_hit', 'mean'),
                 mean_sales=('total_sales', 'mean'))
            .query('n >= @min_count')
            .sort_values('error_rate', ascending=False))


genre_misclassification = misclassification_profile('genre')
console_misclassification = misclassification_profile('console')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
genre_misclassification[['error_rate', 'hit_rate']].plot.barh(ax=axes[0], width=0.8)
axes[0].invert_yaxis()
axes[0].set_title('Error rate vs hit rate by genre')
axes[0].set_xlabel('Rate')
axes[0].legend(fontsize=8)

decade_profile = (classification_frame
                  .assign(is_error=~classification_frame['is_correct'],
                          decade=(classification_frame['release_year'] // 10 * 10))
                  .groupby('decade')
                  .agg(n=('is_error', 'size'), error_rate=('is_error', 'mean'),
                       hit_rate=('is_hit', 'mean'))
                  .query('n >= 100'))
decade_profile[['error_rate', 'hit_rate']].plot.bar(ax=axes[1], width=0.8)
axes[1].set_title('Error rate vs hit rate by release decade')
axes[1].set_ylabel('Rate')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

display(genre_misclassification.round(4))
display(console_misclassification.head(10).round(4))

#### Interpretation

**Every comparison is significant, and every one points the same way.** Misclassified titles have a
*higher* median critic score (7.9 vs 7.2, p ≈ 1e-24), an *earlier* median release year (2007 vs
2008, p ≈ 9e-23) and a substantially *higher* platform count (3.5 vs 2.0, p ≈ 1e-72). The
platform-count difference is by far the strongest: **multi-platform release is the single best
predictor of the model being wrong**.

**Error rate tracks the hit rate almost perfectly.** In both the genre and decade panels the two bars
move together: Shooter (16.8% hit rate → 10.7% error rate), Action-Adventure (14.0% → 10.6%),
Fighting, Sports and Racing at the top; Strategy (1.2% → 1.2%), Adventure, Puzzle and Visual Novel
(0% → 0%) at the bottom. This is the classification-side echo of the heteroscedasticity found in
Part B, and it has a simple explanation: **the model is only ever wrong near the decision boundary,
and only genres with a meaningful hit rate have rows near that boundary**. In a genre where almost
nothing sells a million copies, "not a hit" is correct essentially always. The apparent competence on
Visual Novel and Strategy is not skill -- it is the base rate.

**Critic-score availability again splits the data in two.** Titles with a critic score have a
**12.6% error rate against 4.7%** for titles without -- and their hit rate is 18.5% against 5.1%.
Same mechanism, same conclusion as B.3: `has_critic_score` selects the commercially significant
subpopulation, which is where all the difficulty lives.

**The 1980s row is the exception worth noting.** Error rate reaches **21.8%** there against a 25.4%
hit rate, on only 142 rows. That segment is severely survivorship-biased -- VGChartz recorded
1980s titles mainly *because* they were notable -- so a quarter of them are million-sellers, the
model has almost no data to learn the era's dynamics, and it is wrong more often than anywhere else.

**Regions prone to misclassification, stated plainly:** multi-platform, critically reviewed,
Western-console titles in high-hit-rate genres (Shooter, Action-Adventure, Sports, Racing, Fighting)
— plus the thin, biased pre-2000 tail, plus Japan-dominated handheld titles as C.3 showed. These are
the same rows that dominated the regression's extreme errors, which is a strong indication that the
two tasks are limited by one underlying cause rather than two.

### C.5 Threshold sensitivity analysis (brief 3.4)

Performance is evaluated across thresholds 0.1, 0.2, ..., 0.9, together with the $F_\beta$ curve,
MCC, and the ROC curve with its AUC.

In [ ]:
def sweep_thresholds(y_true, probabilities, thresholds=np.arange(0.1, 0.95, 0.1)):
    rows = []
    for threshold in thresholds:
        labels = (probabilities >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, labels, labels=[0, 1]).ravel()
        rows.append({
            'Threshold': round(float(threshold), 1),
            'Precision': precision_score(y_true, labels, zero_division=0),
            'Recall': recall_score(y_true, labels, zero_division=0),
            'F1': f1_score(y_true, labels, zero_division=0),
            'F0.5': fbeta_score(y_true, labels, beta=0.5, zero_division=0),
            'F2': fbeta_score(y_true, labels, beta=2, zero_division=0),
            'MCC': matthews_corrcoef(y_true, labels),
            'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        })
    return pd.DataFrame(rows).set_index('Threshold')


threshold_table = sweep_thresholds(y_classification, hit_probability)
display(threshold_table.round(4))

print(f'Best F1  at threshold {threshold_table["F1"].idxmax():.1f}  (F1  = {threshold_table["F1"].max():.4f})')
print(f'Best MCC at threshold {threshold_table["MCC"].idxmax():.1f}  (MCC = {threshold_table["MCC"].max():.4f})')
print(f'Best F2  at threshold {threshold_table["F2"].idxmax():.1f}  (F2  = {threshold_table["F2"].max():.4f})')
print(f'Best F0.5 at threshold {threshold_table["F0.5"].idxmax():.1f} (F0.5 = {threshold_table["F0.5"].max():.4f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

for metric, style in [('Precision', '-o'), ('Recall', '-s'), ('F1', '-^'), ('MCC', '-d')]:
    axes[0].plot(threshold_table.index, threshold_table[metric], style, ms=4, label=metric)
axes[0].axvline(0.5, color='grey', ls=':', lw=1)
axes[0].axvline(threshold_table['MCC'].idxmax(), color='crimson', ls='--', lw=1.2,
                label=f'MCC-optimal = {threshold_table["MCC"].idxmax():.1f}')
axes[0].set_xlabel('Decision threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Metrics as a function of the threshold')
axes[0].legend(fontsize=8)

axes[1].plot(threshold_table.index, threshold_table['FP'], '-o', ms=4, label='False Positives')
axes[1].plot(threshold_table.index, threshold_table['FN'], '-s', ms=4, label='False Negatives')
axes[1].set_xlabel('Decision threshold')
axes[1].set_ylabel('Count')
axes[1].set_yscale('log')
axes[1].set_title('The FP / FN trade-off')
axes[1].legend(fontsize=8)

beta_values = np.linspace(0.1, 5.0, 60)
for threshold in [0.1, 0.2, 0.3, 0.5, 0.7]:
    labels = (hit_probability >= threshold).astype(int)
    axes[2].plot(beta_values,
                 [fbeta_score(y_classification, labels, beta=b, zero_division=0) for b in beta_values],
                 lw=1.8, label=f'threshold = {threshold}')
axes[2].axvline(1.0, color='black', ls=':', lw=1)
axes[2].text(1.05, 0.05, 'beta = 1 (F1)', fontsize=8)
axes[2].set_xlabel('beta  (weight on recall relative to precision)')
axes[2].set_ylabel('F-beta score')
axes[2].set_title('F-beta as a function of beta')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

for name, probabilities in classification_oof.items():
    false_positive_rate, true_positive_rate, _ = roc_curve(y_classification, probabilities)
    axes[0].plot(false_positive_rate, true_positive_rate, lw=1.8,
                 label=f'{name} (AUC = {roc_auc_score(y_classification, probabilities):.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='random (AUC = 0.5)')
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].set_title('ROC curves')
axes[0].legend(fontsize=8, loc='lower right')

for name, probabilities in classification_oof.items():
    precision_values, recall_values, _ = precision_recall_curve(y_classification, probabilities)
    axes[1].plot(recall_values, precision_values, lw=1.8,
                 label=f'{name} (AP = {average_precision_score(y_classification, probabilities):.4f})')
axes[1].axhline(y_classification.mean(), color='black', ls='--', lw=1,
                label=f'base rate = {y_classification.mean():.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall curves (the informative view under imbalance)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

#### Interpretation

**The trade-off is steep and highly asymmetric.** Moving the threshold from 0.1 to 0.9 takes
precision from **0.34 to 0.96** while recall collapses from **0.76 to 0.04**. In absolute counts,
false positives fall from 2,265 to 3 while false negatives climb from 360 to 1,438. The default
threshold of 0.5 is nowhere near optimal for any balanced criterion -- it sits deep in the
precision-favouring half of the curve purely as an artefact of the 8% base rate.

**Where each criterion places the operating point:**

| Criterion | Optimal threshold | What it implies |
|---|---|---|
| $F_2$ (recall-weighted) | **0.1** | discovery / screening: catch 76% of hits, accept 2,265 false alarms |
| $F_1$ (balanced) | **0.2** | 0.47 precision at 0.61 recall -- the best single compromise |
| MCC (balance-aware) | **0.2** | agrees with $F_1$; MCC = 0.49 |
| $F_{0.5}$ (precision-weighted) | **0.4** | investment decisions: 0.65 precision, only 335 false alarms |

**The $F_\beta$ curves make the dependence explicit.** Each curve crosses the others: at low $\beta$
(precision-dominant) the high-threshold curves win; as $\beta$ grows past ~1 the ordering inverts and
the 0.1-threshold curve dominates. There is no threshold that is best for all $\beta$ -- **the choice
of operating point is a business decision, not a statistical one**, and this plot is the artefact
that hands that decision to the stakeholder rather than hiding it inside a default.

**Stable and unstable operating regions.**

* **Stable: thresholds 0.2 to 0.4.** F1 stays within a narrow band (0.503-0.532) and MCC within
  0.484-0.490 across this whole range. A model deployed anywhere in this window will behave
  predictably even if the score distribution drifts slightly. This region is the recommended
  operating range.
* **Unstable: below 0.2.** Precision falls off a cliff (0.47 → 0.34 for one step) as thousands of
  low-probability negatives are swept in. Small changes in threshold or in the input distribution
  produce large swings in the false-positive count.
* **Unstable: above 0.6.** Recall collapses towards zero and the positive-prediction count becomes so
  small (66 predictions at 0.9, of which 3 are false positives) that precision is estimated from a
  handful of rows and is extremely sensitive to the sample. This region is also where averaging-based
  ensembles degenerate: a bagged forest's probabilities are bounded by the vote share of its trees and
  rarely approach 1, so a high enough threshold can return *no* positive predictions at all and leave
  every metric undefined.

**ROC vs PR -- the most important methodological point in Part C.** The ROC curves for the two
ensembles look almost perfect (AUC ≈ 0.90) and are visually near-indistinguishable. The PR curves
tell a different story: average precision is only **~0.56**, and while that is roughly **7× the
0.08 base rate** -- a genuinely strong result -- it is nowhere near the impression the ROC gives.
The reason is structural: ROC's x-axis is the false positive rate, normalised by 17,325 negatives,
so 1,000 false positives barely register. Under class imbalance the PR curve is the honest picture,
and it is also where the models actually separate: Gradient Boosting beats Random Forest on PR-AUC
while *losing* marginally on ROC-AUC.

### C.6 Do the regression and classification errors fail on the same rows?

Parts B and C were run independently. If the two tasks are limited by the same underlying cause,
their errors should overlap far more than chance allows. This is testable.

In [ ]:
regression_tail = absolute_errors >= extreme_cutoff        # top 5% of regression |errors|
classification_error = ~is_correct                          # misclassified by the classifier

overlap = pd.crosstab(pd.Series(regression_tail, name='In regression top-5% error'),
                      pd.Series(classification_error, name='Misclassified'))
display(overlap)

expected_overlap = regression_tail.mean() * classification_error.mean() * len(games)
observed_overlap = int((regression_tail & classification_error).sum())
chi2, p_value, _, _ = stats.chi2_contingency(overlap)

print(f'Rows in both error sets : {observed_overlap:,}')
print(f'Expected if independent : {expected_overlap:.0f}')
print(f'Lift over chance        : {observed_overlap / expected_overlap:.1f}x')
print(f'Chi-squared test of independence: chi2 = {chi2:.1f}, p = {p_value:.2e}')
print()
print(f'P(misclassified | in regression error tail) = '
      f'{classification_error[regression_tail].mean():.1%}')
print(f'P(misclassified | not in tail)              = '
      f'{classification_error[~regression_tail].mean():.1%}')

The overlap is **~10x** what independence would predict, and the chi-squared test rejects
independence decisively. **66.5%** of the rows in the regression error tail are also misclassified,
against **3.3%** of the rows outside it -- a twenty-fold difference. **The two tasks are not failing for two different
reasons -- they are failing on the same games, for the same reason:** the feature set contains no
representation of franchise strength, marketing, or regional market structure, so both models fall
back on the conditional average for exactly the titles whose value lies in departing from it.

### C.7 Classification discussion (brief 3.5)

**Strengths and limitations relative to the data.** The tree ensembles are well matched to this
dataset: PR-AUC ≈ 0.56 against a 0.08 base rate is a ~7× lift, achieved from release metadata alone,
with no sales-derived inputs. They handle the mixed numeric/high-cardinality-categorical design
without manual interaction terms, tolerate 78% missingness in `critic_score` through the
missingness-indicator pattern, and produce probabilities that are close to calibrated
(Brier ≈ 0.050). The limitations are equally clear: recall at the default threshold is 0.32; the
linear and single-tree baselines are materially worse (ROC-AUC 0.84 and 0.81), showing the signal is
interaction-driven; and no model exceeds ROC-AUC 0.90, a ceiling that four very different algorithms
all hit — which is the signature of an information limit rather than a capacity limit.

**Key failure modes.**

1. **Threshold mis-specification.** At 0.5 the model produces 5.3× more false negatives than false
   positives. This is the *cheapest* failure to fix — it needs no retraining, only a threshold moved
   to the 0.2-0.4 stable region (or `class_weight='balanced'` at training time).
2. **Confident false negatives.** 58% of missed hits carry P(hit) ≤ 0.20. Unlike (1), this cannot be
   fixed by a threshold, because the model places these rows deep in the negative region.
3. **Regional blindness.** Japan-dominated titles are systematically missed. HW1 measured Japan's
   decoupling ($r \approx 0.07\text{–}0.21$ with Western regions); the classifier reproduces that
   blind spot exactly, because nothing in the feature set encodes a title's target market.
4. **Sparse historical strata.** The 1980s segment carries a 21.8% error rate on 142 survivorship-biased
   rows.
5. **Business-model coverage gaps.** MMOs, episodic releases and digital-first titles appear too
   rarely for the model to learn them, and the target itself (physical retail sales) misrepresents them.

**Assumptions vs observed performance.** Logistic Regression assumes the log-odds of a hit are linear
and additive in the features; the 0.06 ROC-AUC gap to the ensembles is the price of that assumption.
Every model assumes i.i.d. rows, which is false here (same title across consoles, franchise
clustering), so all reported scores are mildly optimistic. The default 0.5 threshold implicitly
assumes balanced classes and equal error costs — both false — which is the direct cause of failure
mode (1). And all models assume the training distribution transfers: the 1980s and Japan-dominated
results show that it does not, uniformly.

**Most significant insights.** First, **the metric determines the conclusion**: ROC-AUC 0.90 and
PR-AUC 0.56 describe the same model, and only the second is honest under 8% imbalance. Second,
**high confidence is not high reliability in the direction that matters** — the model's misses are
confident, so uncertainty-based safety nets catch false positives but not false negatives. Third,
**the classification and regression failures are the same failure** (section C.6), which means one
fix — better features — addresses both.

**Recommendations for improvement.**

* **Operating point.** Deploy at a threshold in the 0.2-0.4 stable band, chosen by the stakeholder's
  own FP/FN cost ratio, rather than at the 0.5 default. Alternatively train with
  `class_weight='balanced'`, which moves the same trade-off into the loss function.
* **Features.** Add a franchise/series identifier and sequel index (extractable from the title string),
  a first-party-publisher flag, a publisher's historical hit rate (target-encoded inside the folds),
  and an is-re-release/special-edition flag. These target failure modes (2) and (5) directly.
* **Regional structure.** Model the Japanese market separately, or add a Japanese-publisher/developer
  indicator, to address failure mode (3).
* **Validation.** Replace `StratifiedKFold` with a grouped split on title or franchise, so the reported
  scores measure generalisation to genuinely new titles.
* **Data hygiene.** Remove or flag the zero-sales re-release rows and the partial-period rows
  identified in B.4; they are mislabeled ground truth, and no model can learn around them.

---

## Part D -- Final Reflection (brief section 4)

### 1. Where does the model fail most?

Both models fail in the same place, and section C.6 confirmed statistically that it is literally the
same rows. The failure region is the **commercially significant segment**: multi-platform, critically
reviewed, Western-console titles in high-hit-rate genres (Shooter, Action-Adventure, Sports, Fighting,
Racing). Concretely:

* the regression's absolute error grows **7×** from the lowest to the highest decile of predicted
  sales, and **81% of its extreme errors are under-predictions** of blockbusters;
* the classifier's error rate tracks the genre hit rate almost one-for-one, reaching 10-11% in Shooter
  and Action-Adventure against ~0% in Visual Novel and Strategy;
* both fail hardest on **multi-platform titles** (regression MAE 0.109 → 0.210 from one platform to six;
  median platform count 2.0 for correct vs 3.5 for incorrect classifications);
* two smaller but distinct failure regions: **Japan-dominated titles**, and the thin, survivorship-biased
  **pre-2000 catalogue**.

The unifying statement: **error magnitude tracks commercial scale and its variance.** The models are
accurate about the long tail of games that were never going to sell, and unreliable about precisely
the titles anyone would build such a model to identify.

### 2. Are the failures due to data, model, or formulation?

All three, in that order of importance.

**Data (the dominant cause).** The individual extreme-error rows in B.4 are the clearest evidence.
The target is *inconsistently defined*: `Resident Evil 5: Gold Edition` records 0.00M because it is a
re-release; `Undertale (PS4)` records 0.01M because VGChartz tracks physical retail and Undertale sold
digitally; one row is literally titled `... (US weekly sales)` inside a lifetime-sales column. Beyond
that, `critic_score` is missing for 78% of the modelling frame, `total_sales` itself is missing for
70% of the catalogue in a structurally biased way, and the pre-2000 era is survivorship-filtered.
The label is noisy, and no algorithm learns around a noisy label.

**Model formulation / features (a close second).** The feature set contains nothing about **franchise
identity, marketing spend, pre-order volume, sequel position, or target market**. `GTA V (X360)` was
predicted at 0.6M against 15.9M actual because "Rockstar Games, Action, 2013, X360" genuinely
describes a game that usually sells a few hundred thousand copies. The model is behaving correctly
given the information it has. The evidence that this is a *feature* limit and not an *algorithm*
limit is that six very different regression algorithms and four classifiers all plateau at the same
place and fail on the same rows.

**Problem formulation (real but smallest).** Squared error on a log target optimises the conditional
mean, which in a power-law market systematically under-predicts the tail — visible as the positive
residual skew across every model. The 1.0M hit threshold, though industry-standard, imposes a hard
boundary on a continuous quantity, so a 0.99M title and a 1.01M title are treated as categorically
different. And the default 0.5 decision threshold is simply wrong for an 8% base rate.

### 3. What improvements would I propose?

**Data quality (highest leverage, lowest cost).** Drop or flag the ~7% of rows with
`total_sales = 0`; parse the title string to flag re-releases, GOTY/Gold/Complete editions and
partial-period rows; treat `'Unknown'` publisher/developer as missing throughout (already done here).

**Feature engineering.** A franchise key and sequel index from the title string; a first-party
publisher flag (Nintendo on Nintendo hardware, Sony on PlayStation); a publisher's historical hit
rate, target-encoded strictly inside the training folds; a Japanese-publisher/developer indicator to
address the regional blind spot; console-lifecycle position (years since that console launched);
and an is-digital-era flag interacting with release year.

**Modelling.** Replace `KFold`/`StratifiedKFold` with a **grouped split on title or franchise**, which
is the single most important methodological fix — current scores are optimistic because franchise
siblings leak across folds. Consider quantile regression or a Tweedie/gamma objective to model the
tail explicitly rather than regressing toward the mean; and report prediction *intervals* rather than
point estimates, since the heteroscedasticity result means a single number is misleading by
construction.

**Deployment.** Choose the operating threshold from the stable 0.2-0.4 band according to the actual
FP/FN cost ratio, and route low-confidence predictions to human review — with the explicit caveat
from C.3 that this catches false positives, not the confident false negatives.

### 4. What insights did I gain?

**Error analysis is a different activity from model evaluation, and it is the more informative one.**
The summary table said Gradient Boosting and Random Forest were nearly tied on MAE. The error
analysis showed they differ in tail behaviour, that both are biased in the same direction, and that
the bias has a name (regression toward the mean in a power-law market). None of that is visible in a
metric.

**A good metric can conceal a bad model.** ROC-AUC 0.90 and PR-AUC 0.56 describe the same
classifier; 92% accuracy is achievable by predicting "never a hit". Choosing the metric *before*
looking at the class balance would have produced a confident and wrong conclusion.

**The most consequential decisions were made before any model was fitted.** Excluding the regional
sales columns took $R^2$ from a meaningless ~1.0 to an honest 0.55. Log-transforming the target
changed which errors the loss function cared about. Deciding that `has_critic_score` was signal rather
than nuisance produced one of the strongest findings in the notebook. The modelling library did the
easy part.

**Residual skew and kurtosis are diagnostic, not decorative.** Skewness ~1.8 said the model is
systematically conservative about blockbusters; excess kurtosis ~8.6 said the RMSE is hostage to a
few dozen rows. Both pointed at the same missing-feature problem, and both were visible before a
single individual error was inspected.

**A view that changed.** HW1 concluded that critic score was a weak correlate of sales and treated it
as a minor variable. This assignment showed something sharper: critic score is not weakly informative
*uniformly* — it is nearly useless above a score of ~7, and its very *availability* is a stronger
signal than its value, because being reviewed at all is a proxy for commercial significance. A
feature's missingness pattern carried more information than the feature itself, which is not a
possibility I would have taken seriously before running the error analysis.

**Finally, the honest scope of the result.** Everything above is conditional on a population of
titles that VGChartz chose to track, measured by physical retail sales, in a market that has since
moved substantially to digital distribution. The model does not predict "how well a game will sell".
It predicts "how well VGChartz will record a game as having sold", and the difference between those
two statements is where several of its largest errors live.